In [3]:
"""
台股BIGRU預測系統 - 基準模型
用於與HFSLS-PSO-BIGRU進行消融實驗對比
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 30
loss = "mse"
measure = [[]]
true_pre = [[], []]

# ============================================================
# 進度追蹤器
# ============================================================
class ProgressTracker:
    def __init__(self, total_stocks):
        self.total_stocks = total_stocks
        self.start_time = time.time()
        self.stock_start_time = None
        
    def start_stock(self, stock_name, stock_idx):
        self.stock_start_time = time.time()
        print(f"\n{'='*70}")
        print(f"📈 [{stock_idx}/{self.total_stocks}] 處理：{stock_name}")
        print(f"{'='*70}")
    
    def end_stock(self, stock_name):
        stock_time = time.time() - self.stock_start_time
        print(f"\n{'='*70}")
        print(f"✅ {stock_name} 完成！耗時 {stock_time/60:.1f} 分鐘")
        print(f"{'='*70}")

tracker = None

# ============================================================
# 核心函數
# ============================================================

def process_data(X):
    """轉換為滑動窗口格式"""
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window:
            x.append(list(que))
    return np.array(x)

def build_baseline_bigru(input_shape, units=64, dropout=0.2):
    """
    構建基準BIGRU模型
    使用固定參數（不使用PSO優化）
    """
    model = Sequential([
        Bidirectional(GRU(units, activation='relu', return_sequences=False), 
                     input_shape=input_shape),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def train_baseline_model(x_train, y_train, x_test, y_test, test_indices, dates, stock_name):
    """
    訓練基準BIGRU模型
    不使用PSO優化，使用固定的超參數
    """
    print(f"  🏗️  構建基準BIGRU模型...")
    print(f"  📊 固定參數：units=64, dropout=0.2, epochs=30")
    
    # 使用固定參數構建模型
    model = build_baseline_bigru(
        input_shape=x_train.shape[1:],
        units=64,      # 固定單元數
        dropout=0.2    # 固定dropout率
    )
    
    # 訓練模型
    print(f"  🔄 開始訓練...")
    history = model.fit(
        x_train, y_train, 
        batch_size=32, 
        epochs=30,
        validation_split=0.1,
        verbose=0,
        shuffle=False
    )
    
    # 預測與評估
    y_pred = model.predict(x_test, verbose=0)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"  📊 MSE={mse:.6f}, R²={r2:.4f}")
    
    # 保存結果
    y_pred_inv = scaler.inverse_transform(y_pred)
    y_test_inv = scaler.inverse_transform(y_test)
    
    measure[0] = [mse, rmse, mae, mape, r2]
    true_pre[0] = list(y_test_inv[:, 0])
    true_pre[1] = list(y_pred_inv[:, 0])
    
    # 保存預測結果
    base_dates = [dates[idx] for idx in test_indices]
    predict_dates = [d + pd.Timedelta(days=15) for d in base_dates]
    
    data = pd.DataFrame({
        'stock_name': stock_name,
        'base_date': base_dates,
        'predict_date': predict_dates,
        'predicted_close': y_pred_inv[:, 0],
        'actual_close': y_test_inv[:, 0],
        'error': y_pred_inv[:, 0] - y_test_inv[:, 0],
        'error_pct': ((y_pred_inv[:, 0] - y_test_inv[:, 0]) / y_test_inv[:, 0] * 100)
    })
    
    output_path = 'D:/pythonProject/a-lunwen/data/predicted_baseline_bigru.csv'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    try:
        existing = pd.read_csv(output_path)
        # 刪除該股票的舊數據
        existing = existing[existing['stock_name'] != stock_name]
        existing = pd.concat([existing, data], ignore_index=True)
    except FileNotFoundError:
        existing = data
    
    existing.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"  💾 已保存預測結果")
    
    return model

def fit_baseline(df, dates, stock_name):
    """
    基準模型訓練函數
    使用所有158個特徵（不進行特徵選擇）
    """
    print(f"\n📋 數據：{df.shape[0]}樣本 × {df.shape[1]}特徵")
    print(f"  ⚠️  使用全部特徵（無HFSLS特徵選擇）")
    
    # 歸一化特徵
    X = df.iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    
    # 準備標籤
    y = df['Close'].values[:-window+1]
    y = scaler.fit_transform(y.reshape(-1, 1))
    valid_dates = dates[:-window+1].reset_index(drop=True)
    
    # 轉換為滑動窗口格式
    x = process_data(X.values)
    
    # 分割訓練集和測試集
    split_point = int(len(y) * 0.7)
    
    x_train = x[:split_point]
    y_train = y[:split_point]
    
    x_test = x[split_point:]
    y_test = y[split_point:]
    test_indices = list(range(split_point, len(y)))
    
    print(f"  📊 訓練集：{len(x_train)} 樣本")
    print(f"  📊 測試集：{len(x_test)} 樣本")
    
    # 訓練基準模型
    model = train_baseline_model(
        x_train, y_train, 
        x_test, y_test,
        test_indices, valid_dates, 
        stock_name
    )
    
    return model

def main(data, stock_name):
    """主函數"""
    global measure, true_pre
    
    # 重置全局變量
    measure = [[]]
    true_pre = [[], []]
    
    # 讀取數據
    df_full = pd.read_csv(f'D:/pythonProject/a-lunwen/data/{data}')
    
    if 'Date' in df_full.columns:
        dates = pd.to_datetime(df_full['Date'])
        df = df_full.drop(columns=['Date'])
    else:
        dates = pd.date_range('2021-01-01', periods=len(df_full), freq='D')
        df = df_full
    
    df = df.fillna(0).sort_index()
    
    # 訓練基準模型
    fit_baseline(df, dates, stock_name)
    
    # 輸出結果
    print(f"\n{'='*70}")
    print(f"🎉 {stock_name} 基準模型完成！")
    print(f"{'='*70}")
    print(f"📈 性能指標：")
    print(f"   MSE:  {measure[0][0]:.6f}")
    print(f"   RMSE: {measure[0][1]:.6f}")
    print(f"   MAE:  {measure[0][2]:.6f}")
    print(f"   MAPE: {measure[0][3]:.4f}")
    print(f"   R²:   {measure[0][4]:.4f}")
    print(f"{'='*70}\n")

# ============================================================
# 主程序
# ============================================================
if __name__ == '__main__':
    print("="*70)
    print("🚀 台股BIGRU預測系統 - 基準模型")
    print("   模型：基礎BIGRU（無HFSLS、無PSO）")
    print("   特徵：Alpha158全部特徵（158個）")
    print("   預測：11個交易日後的收盤價")
    print("="*70)
    print("📌 用於消融實驗對比")
    print("="*70)
    
    stock_list = ['台積電', '長榮', '聯發科', '統一超']
    tracker = ProgressTracker(len(stock_list))
    
    total_start = time.time()
    success = 0
    
    # 保存所有股票的結果用於對比
    all_results = []
    
    for idx, stock in enumerate(stock_list, 1):
        try:
            tracker.start_stock(stock, idx)
            main(f"{stock}_date.csv", stock)
            tracker.end_stock(stock)
            
            # 記錄結果
            all_results.append({
                'stock': stock,
                'MSE': measure[0][0],
                'RMSE': measure[0][1],
                'MAE': measure[0][2],
                'MAPE': measure[0][3],
                'R²': measure[0][4]
            })
            
            success += 1
            
        except Exception as e:
            print(f"\n❌ {stock} 失敗：{e}")
            import traceback
            traceback.print_exc()
    
    # 輸出總結
    print(f"\n{'='*70}")
    print(f"🏁 基準模型訓練完成！")
    print(f"{'='*70}")
    print(f"✅ 成功：{success}/{len(stock_list)}")
    print(f"⏱️  總耗時：{(time.time()-total_start)/60:.1f} 分鐘")
    print(f"{'='*70}")
    
    # 輸出性能對比表格
    if all_results:
        print(f"\n📊 基準模型性能總結")
        print(f"{'='*70}")
        print(f"{'股票':<10} {'MSE':<12} {'RMSE':<12} {'R²':<10}")
        print(f"{'-'*70}")
        
        avg_mse = 0
        avg_rmse = 0
        avg_r2 = 0
        
        for result in all_results:
            print(f"{result['stock']:<10} {result['MSE']:<12.6f} {result['RMSE']:<12.6f} {result['R²']:<10.4f}")
            avg_mse += result['MSE']
            avg_rmse += result['RMSE']
            avg_r2 += result['R²']
        
        print(f"{'-'*70}")
        print(f"{'平均':<10} {avg_mse/len(all_results):<12.6f} {avg_rmse/len(all_results):<12.6f} {avg_r2/len(all_results):<10.4f}")
        print(f"{'='*70}")
    
    print(f"\n💾 預測結果：D:/pythonProject/a-lunwen/data/predicted_baseline_bigru.csv")
    print(f"📌 請使用完整模型進行對比實驗")
    print(f"{'='*70}")

🚀 台股BIGRU預測系統 - 基準模型
   模型：基礎BIGRU（無HFSLS、無PSO）
   特徵：Alpha158全部特徵（158個）
   預測：11個交易日後的收盤價
📌 用於消融實驗對比

📈 [1/4] 處理：台積電

📋 數據：1141樣本 × 159特徵
  ⚠️  使用全部特徵（無HFSLS特徵選擇）
  📊 訓練集：778 樣本
  📊 測試集：334 樣本
  🏗️  構建基準BIGRU模型...
  📊 固定參數：units=64, dropout=0.2, epochs=30
  🔄 開始訓練...
  📊 MSE=0.025749, R²=-0.7025
  💾 已保存預測結果

🎉 台積電 基準模型完成！
📈 性能指標：
   MSE:  0.025749
   RMSE: 0.160465
   MAE:  0.150630
   MAPE: 0.1971
   R²:   -0.7025


✅ 台積電 完成！耗時 0.5 分鐘

📈 [2/4] 處理：長榮

📋 數據：1141樣本 × 159特徵
  ⚠️  使用全部特徵（無HFSLS特徵選擇）
  📊 訓練集：778 樣本
  📊 測試集：334 樣本
  🏗️  構建基準BIGRU模型...
  📊 固定參數：units=64, dropout=0.2, epochs=30
  🔄 開始訓練...
  📊 MSE=0.006310, R²=0.2665
  💾 已保存預測結果

🎉 長榮 基準模型完成！
📈 性能指標：
   MSE:  0.006310
   RMSE: 0.079434
   MAE:  0.065257
   MAPE: 0.0827
   R²:   0.2665


✅ 長榮 完成！耗時 0.5 分鐘

📈 [3/4] 處理：聯發科

📋 數據：1141樣本 × 159特徵
  ⚠️  使用全部特徵（無HFSLS特徵選擇）
  📊 訓練集：778 樣本
  📊 測試集：334 樣本
  🏗️  構建基準BIGRU模型...
  📊 固定參數：units=64, dropout=0.2, epochs=30
  🔄 開始訓練...
  📊 MSE=0.006894, R²=0.5443
  💾 已保存預測結果

🎉 聯發科 基準模型完成！
📈 性